# Clinical Comorbidity Pattern Mining
**Course deliverable — Association rule mining over a clinical comorbidity dataset.**

This notebook reproduces the analysis that powers the
[Clinical Comorbidity Dashboard](https://clinical-comorbidity-dashboard.streamlit.app).
It performs frequent-itemset mining with **FP-Growth** and derives
**association rules** (support / confidence / lift) that surface clinically
meaningful co-occurrence patterns between primary and secondary diagnoses.

## Contents
1. Problem definition
2. Environment setup
3. Data loading & inspection
4. Preprocessing & transaction encoding
5. Frequent itemset mining (FP-Growth)
6. Association rule generation
7. Pattern visualization
8. Findings & conclusions


## 1. Problem definition

Hospitals collect rich diagnostic records but rarely surface the *patterns*
between conditions that travel together. We mine the dataset for
**comorbidity patterns** — combinations of diagnoses that co-occur more
often than chance — to support:

- Earlier screening (if A then likely B)
- Multi-disciplinary care planning
- Resource forecasting

We use **FP-Growth** instead of Apriori because it scans the data twice
and avoids combinatorial candidate generation, which matters at scale.


## 2. Environment setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
plt.rcParams["figure.dpi"] = 110

## 3. Data loading & inspection

The dataset is versioned with **Dolt** (Git-for-data). The local working copy
lives under `data_raw/`. Each row represents a patient encounter with one
primary and one or more secondary diagnoses.

In [ ]:
raw = pd.read_csv("../data_raw/raw_data.csv")
print("Shape:", raw.shape)
raw.head()

In [ ]:
raw.info()
raw.describe(include="all").T.head(10)

## 4. Preprocessing & transaction encoding

Each patient record becomes a **transaction** — a set of diagnoses.
We collapse primary + secondary diagnoses per patient into one set, then
one-hot encode with `TransactionEncoder` so FP-Growth can consume it.

In [ ]:
transactions_df = pd.read_csv("../transactions/transactions.csv")
transactions = (
    transactions_df.groupby("patient_id")["diagnosis"].apply(list).tolist()
)
print(f"{len(transactions):,} transactions")
print("Example:", transactions[0][:6])

In [ ]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
encoded = pd.DataFrame(te_array, columns=te.columns_)
print("Encoded matrix:", encoded.shape)
encoded.iloc[:3, :8]

## 5. Frequent itemset mining — FP-Growth

We mine itemsets with **min_support = 0.01** (1% of patients).
This is low enough to surface rare-but-clinically-relevant combinations
while excluding noise.

In [ ]:
itemsets = fpgrowth(encoded, min_support=0.01, use_colnames=True)
itemsets["length"] = itemsets["itemsets"].apply(len)
itemsets = itemsets.sort_values("support", ascending=False).reset_index(drop=True)
print(f"{len(itemsets):,} frequent itemsets")
itemsets.head(15)

### Itemset size distribution

In [ ]:
itemsets["length"].value_counts().sort_index()

## 6. Association rules

From the itemsets we generate rules with **min_confidence = 0.5** and
**lift > 1** (positively correlated). Lift is the headline metric — it
tells us how much more likely B is given A versus B's base rate.

In [ ]:
rules = association_rules(itemsets, metric="confidence", min_threshold=0.5)
rules = rules[rules["lift"] > 1].sort_values("lift", ascending=False).reset_index(drop=True)
print(f"{len(rules):,} rules")
rules[["antecedents","consequents","support","confidence","lift"]].head(15)

### Headline metrics

In [ ]:
print(f"Total rules:           {len(rules):,}")
print(f"Median confidence:     {rules['confidence'].median():.3f}")
print(f"Median lift:           {rules['lift'].median():.3f}")
print(f"Max lift:              {rules['lift'].max():.3f}")
print(f"Strong rules (lift>3): {(rules['lift'] > 3).sum():,}")

## 7. Pattern visualization

### 7.1 Confidence vs Lift (bubble = support)
Top-right corner = clinically actionable rules: high confidence *and* high lift.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(rules["confidence"], rules["lift"],
                s=rules["support"]*4000, c=rules["lift"],
                cmap="viridis", alpha=0.6, edgecolors="white")
ax.set_xlabel("Confidence")
ax.set_ylabel("Lift")
ax.set_title("Association rules — Confidence vs Lift (size = support)")
plt.colorbar(sc, ax=ax, label="Lift")
plt.tight_layout(); plt.show()

### 7.2 Top 15 rules by lift

In [ ]:
top = rules.head(15).copy()
top["rule"] = top.apply(
    lambda r: f"{', '.join(list(r['antecedents'])[:2])} → {', '.join(list(r['consequents'])[:2])}",
    axis=1,
)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["rule"][::-1], top["lift"][::-1], color="#3b82f6")
for i, (l, c) in enumerate(zip(top["lift"][::-1], top["confidence"][::-1])):
    ax.text(l + 0.05, i, f"conf={c:.2f}", va="center", fontsize=8, color="#475569")
ax.set_xlabel("Lift"); ax.set_title("Top 15 rules by lift")
plt.tight_layout(); plt.show()

### 7.3 Top 15 frequent itemsets by support

In [ ]:
top_i = itemsets.head(15).copy()
top_i["label"] = top_i["itemsets"].apply(lambda s: ", ".join(list(s)[:3]))
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_i["label"][::-1], top_i["support"][::-1], color="#7c3aed")
ax.set_xlabel("Support"); ax.set_title("Top 15 frequent itemsets")
plt.tight_layout(); plt.show()

## 8. Findings & conclusions

- **FP-Growth scaled cleanly** at 1% support — Apriori would have generated
  orders of magnitude more candidates on the same data.
- The **top rules by lift** point to well-established clinical comorbidities
  (e.g. cardiovascular ↔ metabolic clusters), which validates the pipeline,
  while a long tail of mid-lift rules surfaces less obvious pairings worth
  clinician review.
- **Confidence alone is misleading** when a consequent is very common — lift
  is the right gate, which is why the dashboard sorts by lift by default.
- The **Streamlit dashboard** wraps this same pipeline behind interactive
  filters (min_support / min_confidence / diagnosis pickers) so clinicians
  can re-run the analysis without touching a notebook.

### Reproducibility
Raw data is versioned with **Dolt**. To reproduce:
```bash
dolt clone <remote> clinical-data
dolt sql -q "SELECT * FROM encounters" -r csv > data_raw/raw_data.csv
jupyter nbconvert --to notebook --execute notebooks/clinical_comorbidity_analysis.ipynb
```
